In [ ]:
import matplotlib.pyplot as plt
from sklearn import datasets
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import numpy as np
import os

iris = datasets.load_iris()
x = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names
target = iris.target

In [ ]:
# Conjunto de datos original
original = pd.DataFrame(data=x, columns=feature_names)
original['target'] = target

# Conjunto de datos estandarizado
x_scaled = StandardScaler().fit_transform(x)
estandarizados = pd.DataFrame(data=x_scaled, columns=feature_names)
estandarizados['target'] = target

# Conjunto de datos normalizado
x_minmax = MinMaxScaler().fit_transform(x)
normalizados = pd.DataFrame(data=x_minmax, columns=feature_names)
normalizados['target'] = target

# Creamos objetos PCA reutilizables
pca_95 = PCA(n_components=0.95)
pca_80 = PCA(n_components=0.80)

# PCA 95% sobre datos originales
x_pca95_original = pca_95.fit_transform(x)
columnas_pca95_original = [f'PC{i+1}' for i in range(x_pca95_original.shape[1])]
originalPCA95 = pd.DataFrame(data=x_pca95_original, columns=columnas_pca95_original)
originalPCA95['target'] = target

# PCA 80% sobre datos originales
x_pca80_original = pca_80.fit_transform(x)
columnas_pca80_original = [f'PC{i+1}' for i in range(x_pca80_original.shape[1])]
originalPCA80 = pd.DataFrame(data=x_pca80_original, columns=columnas_pca80_original)
originalPCA80['target'] = target

# PCA 95% sobre datos estandarizados
x_pca95_estandarizado = pca_95.fit_transform(x_scaled)
columnas_pca95_estandarizado = [f'PC{i+1}' for i in range(x_pca95_estandarizado.shape[1])]
estandarizadoPCA95 = pd.DataFrame(data=x_pca95_estandarizado, columns=columnas_pca95_estandarizado)
estandarizadoPCA95['target'] = target

# PCA 80% sobre datos estandarizados
x_pca80_estandarizado = pca_80.fit_transform(x_scaled)
columnas_pca80_estandarizado = [f'PC{i+1}' for i in range(x_pca80_estandarizado.shape[1])]
estandarizadoPCA80 = pd.DataFrame(data=x_pca80_estandarizado, columns=columnas_pca80_estandarizado)
estandarizadoPCA80['target'] = target

# PCA 95% sobre datos normalizados
x_pca95_normalizado = pca_95.fit_transform(x_minmax)
columnas_pca95_normalizado = [f'PC{i+1}' for i in range(x_pca95_normalizado.shape[1])]
normalizadoPCA95 = pd.DataFrame(data=x_pca95_normalizado, columns=columnas_pca95_normalizado)
normalizadoPCA95['target'] = target

# PCA 80% sobre datos normalizados
x_pca80_normalizado = pca_80.fit_transform(x_minmax)
columnas_pca80_normalizado = [f'PC{i+1}' for i in range(x_pca80_normalizado.shape[1])]
normalizadoPCA80 = pd.DataFrame(data=x_pca80_normalizado, columns=columnas_pca80_normalizado)
normalizadoPCA80['target'] = target

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Configuración de validación cruzada
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Diccionario con todos los conjuntos de datos
datasets = {
    'original': original,
    'estandarizados': estandarizados,
    'normalizados': normalizados,
    'originalPCA95': originalPCA95,
    'originalPCA80': originalPCA80,
    'estandarizadoPCA95': estandarizadoPCA95,
    'estandarizadoPCA80': estandarizadoPCA80,
    'normalizadoPCA95': normalizadoPCA95,
    'normalizadoPCA80': normalizadoPCA80
}

# Crear carpeta para guardar particiones
output_dir = 'particiones_cv'
os.makedirs(output_dir, exist_ok=True)

# Generar particiones para cada conjunto de datos
for dataset_name, dataset in datasets.items():
    print(f"Generando particiones para {dataset_name}...")
    
    # Crear carpeta específica para este dataset
    dataset_dir = os.path.join(output_dir, dataset_name)
    os.makedirs(dataset_dir, exist_ok=True)
    
    # Separar características y target
    X = dataset.drop('target', axis=1)
    y = dataset['target']
    
    # Generar las 5 particiones
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        # Crear carpeta para este fold
        fold_dir = os.path.join(dataset_dir, f'fold_{fold}')
        os.makedirs(fold_dir, exist_ok=True)
        
        # Separar train y test
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Guardar en CSV
        X_train.to_csv(os.path.join(fold_dir, 'X_train.csv'), index=False)
        X_test.to_csv(os.path.join(fold_dir, 'X_test.csv'), index=False)
        y_train.to_csv(os.path.join(fold_dir, 'y_train.csv'), index=False, header=True)
        y_test.to_csv(os.path.join(fold_dir, 'y_test.csv'), index=False, header=True)
        
        print(f"  Fold {fold}: Train={len(X_train)}, Test={len(X_test)}")

print("\n✓ Particiones generadas exitosamente en la carpeta 'particiones_cv'")
print(f"✓ Total: {len(datasets)} conjuntos de datos × {n_splits} folds = {len(datasets)*n_splits} conjuntos de particiones")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle
import json

# Definir los modelos a evaluar
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', random_state=42),
    'NaiveBayes': GaussianNB(),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Función para calcular todas las métricas
def calcular_metricas(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall': recall_score(y_true, y_pred, average='weighted'),
        'f1_score': f1_score(y_true, y_pred, average='weighted')
    }

# Estructura para guardar resultados
resultados = {dataset_name: {model_name: [] for model_name in models.keys()} 
              for dataset_name in datasets.keys()}

print("Iniciando validación cruzada...")
print("=" * 80)

In [ ]:
# Crear carpeta para guardar modelos y resultados
models_dir = 'modelos_cv'
os.makedirs(models_dir, exist_ok=True)

# Iterar sobre cada conjunto de datos
for dataset_name in datasets.keys():
    print(f"\n{'='*80}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*80}")
    
    dataset_dir = os.path.join(output_dir, dataset_name)
    
    # Iterar sobre cada fold
    for fold in range(1, n_splits + 1):
        print(f"\n--- Fold {fold}/{n_splits} ---")
        fold_dir = os.path.join(dataset_dir, f'fold_{fold}')
        
        # Cargar datos de entrenamiento y test
        X_train = pd.read_csv(os.path.join(fold_dir, 'X_train.csv'))
        X_test = pd.read_csv(os.path.join(fold_dir, 'X_test.csv'))
        y_train = pd.read_csv(os.path.join(fold_dir, 'y_train.csv')).values.ravel()
        y_test = pd.read_csv(os.path.join(fold_dir, 'y_test.csv')).values.ravel()
        
        # Entrenar y evaluar cada modelo
        for model_name, model in models.items():
            # Entrenar el modelo
            model.fit(X_train, y_train)
            
            # Realizar predicciones
            y_pred = model.predict(X_test)
            
            # Calcular métricas
            metricas = calcular_metricas(y_test, y_pred)
            resultados[dataset_name][model_name].append(metricas)
            
            # Guardar el modelo entrenado
            model_path = os.path.join(models_dir, dataset_name, model_name)
            os.makedirs(model_path, exist_ok=True)
            model_file = os.path.join(model_path, f'fold_{fold}.pkl')
            with open(model_file, 'wb') as f:
                pickle.dump(model, f)
            
            print(f"  {model_name}: F1={metricas['f1_score']:.4f}, Acc={metricas['accuracy']:.4f}")

print(f"\n{'='*80}")
print("✓ Validación cruzada completada")
print(f"✓ Modelos guardados en '{models_dir}'")
print(f"{'='*80}")

In [ ]:
# Calcular y mostrar estadísticas finales (media y desviación típica)
print("\n" + "="*80)
print("RESULTADOS FINALES - VALIDACIÓN CRUZADA (5 FOLDS)")
print("="*80)

# Crear estructura para guardar resultados finales
resultados_finales = {}

for dataset_name in datasets.keys():
    print(f"\n{dataset_name}:")
    print("-" * 80)
    resultados_finales[dataset_name] = {}
    
    for model_name in models.keys():
        # Extraer todas las métricas de los 5 folds
        metricas_folds = resultados[dataset_name][model_name]
        
        # Calcular media y desviación típica para cada métrica
        accuracy_values = [m['accuracy'] for m in metricas_folds]
        precision_values = [m['precision'] for m in metricas_folds]
        recall_values = [m['recall'] for m in metricas_folds]
        f1_values = [m['f1_score'] for m in metricas_folds]
        
        resultados_finales[dataset_name][model_name] = {
            'accuracy_mean': np.mean(accuracy_values),
            'accuracy_std': np.std(accuracy_values),
            'precision_mean': np.mean(precision_values),
            'precision_std': np.std(precision_values),
            'recall_mean': np.mean(recall_values),
            'recall_std': np.std(recall_values),
            'f1_mean': np.mean(f1_values),
            'f1_std': np.std(f1_values)
        }
        
        print(f"\n  {model_name}:")
        print(f"    Accuracy:  {np.mean(accuracy_values):.4f} ± {np.std(accuracy_values):.4f}")
        print(f"    Precision: {np.mean(precision_values):.4f} ± {np.std(precision_values):.4f}")
        print(f"    Recall:    {np.mean(recall_values):.4f} ± {np.std(recall_values):.4f}")
        print(f"    F1-Score:  {np.mean(f1_values):.4f} ± {np.std(f1_values):.4f}")

# Guardar resultados finales en JSON
resultados_file = os.path.join(models_dir, 'resultados_finales.json')
with open(resultados_file, 'w') as f:
    json.dump(resultados_finales, f, indent=4)

print(f"\n{'='*80}")
print(f"✓ Resultados finales guardados en '{resultados_file}'")
print("="*80)

In [ ]:
# Crear tabla comparativa de F1-Score
print("\n" + "="*80)
print("TABLA COMPARATIVA - F1-SCORE MEDIO")
print("="*80)

# Crear DataFrame para visualización
comparison_data = []
for dataset_name in datasets.keys():
    for model_name in models.keys():
        f1_mean = resultados_finales[dataset_name][model_name]['f1_mean']
        f1_std = resultados_finales[dataset_name][model_name]['f1_std']
        comparison_data.append({
            'Dataset': dataset_name,
            'Modelo': model_name,
            'F1-Score': f1_mean,
            'Std': f1_std
        })

df_comparison = pd.DataFrame(comparison_data)

# Crear tabla pivote para mejor visualización
pivot_table = df_comparison.pivot(index='Dataset', columns='Modelo', values='F1-Score')
print("\n")
print(pivot_table.to_string())

# Encontrar mejor combinación
best_result = df_comparison.loc[df_comparison['F1-Score'].idxmax()]
print(f"\n{'='*80}")
print(f"MEJOR RESULTADO:")
print(f"  Dataset: {best_result['Dataset']}")
print(f"  Modelo: {best_result['Modelo']}")
print(f"  F1-Score: {best_result['F1-Score']:.4f} ± {best_result['Std']:.4f}")
print("="*80)